# Engenharia de Features

Técnicas avançadas de transformação e seleção de features.

**Conceitos:**
- Features polinomiais (`PolynomialFeatures`)
- Seleção de features (`SelectKBest`, `RFE`)
- Transformação de distribuições (`PowerTransformer`)

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import PowerTransformer

In [ ]:
# Gera dados sintéticos não lineares para regressão
m = 100
X = 6 * np.random.rand(m, 1) - 3
y = 0.5 * X**2 + X + 2 + np.random.rand(m, 1)

# Ajusta regressão linear simples nos dados (subajustada, pois relação é quadrática)
lin_reg = LinearRegression()
lin_reg.fit(X, y)
X_novo = np.linspace(-3, 3, 100).reshape(100, 1)
y_novo = lin_reg.predict(X_novo)

## 2. Features Polinomiais

In [ ]:
# Cria features polinomiais de grau 2 (x^0, x^1, x^2) para capturar a relação não linear
poly_feat = PolynomialFeatures(degree=2, include_bias=True)
X_poly = poly_feat.fit_transform(X)

print(f'X original: {X[0]}')
print(f'X transformado: {X_poly[0]}')

In [ ]:
# Treina a regressão linear sobre os dados com features polinomiais
lin_reg_poly = LinearRegression()
lin_reg_poly.fit(X_poly, y)

# Transforma os pontos de predição e faz as previsões com o modelo polinomial
X_novo_poly = poly_feat.fit_transform(X_novo)
y_novo_poly = lin_reg_poly.predict(X_novo_poly)

In [ ]:
# Plota os dados reais, a regressão linear simples (reta) e a polinomial (curva)
plt.scatter(X, y, c='blue', label='Dados reais')
plt.plot(X_novo, y_novo, 'r--', label='Regressão Linear(reta)', linewidth=2)
plt.plot(X_novo, y_novo_poly, 'g-', label='Regressão Polinomial(curva)', linewidth=2)
plt.legend()
plt.show

## 3. Seleção de Features com SelectKBest

In [ ]:
# Cria dataset de classificação com 20 features, das quais apenas 3 são informativas
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=3,
    n_redundant=2,
    n_repeated=0,
    n_classes=2,
    random_state=42
)

# Divide em treino e teste (70/30)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)
print(f'Formato original: {X_treino.shape}')

In [ ]:
# Seleciona as 5 melhores features com base no teste F (ANOVA)
select = SelectKBest(score_func=f_classif, k=5)

# Ajusta o seletor nos dados de treino e transforma ambos os conjuntos
X_treino_select = select.fit_transform(X_treino, y_treino)
X_teste_select = select.transform(X_teste)

print(f'Formado após a seleção: {X_treino_select.shape}')

# Recupera qual índice das colunas originais foram mantidas
colunas_select = select.get_support()
print(f'Indice das colunas: {[i for i, x in enumerate(colunas_select) if x]}')

## 4. Seleção com RFE

In [ ]:
# O RFE precisa de um modelo base para julgar a importância das features
modelo_base = RandomForestClassifier(n_estimators=100, random_state=42)

# Seleciona recursivamente as 3 features mais importantes
rfe = RFE(estimator=modelo_base, n_features_to_select=3)
rfe.fit(X_treino, y_treino)

print('Colunas escolhidas pelo RFE:')
# Ranking: 1 significa que foi selecionada; quanto maior o número, menos relevante
for i, col in enumerate(range(X_treino.shape[1])):
  print(f'Coluna {col}: Rank: {rfe.ranking_[i]} {'(Selecionada)' if rfe.support_[i] else ""}')

## 5. Transformação de Distribuições (PowerTransformer)

In [ ]:
# Gera dados com distribuição log-normal (assimétrica à direita)
X_torto = np.random.lognormal(mean=0, sigma=1, size=(1000, 1))

# Aplica PowerTransformer (Yeo-Johnson) para tornar a distribuição mais próxima de uma Gaussiana
pt = PowerTransformer(method='yeo-johnson')
X_normal = pt.fit_transform(X_torto)

# Compara histogramas antes e depois da transformação
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(X_torto, bins=30)
ax[0].set_title('Antes (torto/skewed)')
ax[1].hist(X_normal, bins=30)
ax[1].set_title('Depois (Yeo-Johnson)')
plt.show()